# Import packages

In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import logging
from datetime import datetime
import os
import pickle

from PIL import Image
import traceback

from sklearn.metrics import average_precision_score

from transformers import  ResNetForImageClassification,EfficientNetForImageClassification,AutoModelForImageClassification
 

import torch
from torch.utils.data import Dataset, DataLoader,random_split
from torchvision import transforms

/service/sidekick1/mo1om/miniconda3/envs/SLDL/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuration

In [10]:
config = {
    'batch_size':103,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu', 
    'epochs':15,
    'num_of_classes': 79,
    'patient': 7, # for early stopping
    'learning_rate': 0.001,
    
    'checkpoint': 'microsoft/resnet-152',
    #'checkpoint': 'google/efficientnet-b7',
   '#checkpoint': 'google/vit-base-patch16-224',
    #'checkpoint': 'microsoft/resnet-50'
    'use_valid': False
}

def test_gpu():
    if torch.cuda.is_available():
        print("CUDA is available! GPU is ready to be used.")
        print(f"Device name: {torch.cuda.get_device_name(0)}")
    else:
        print("CUDA is not available. GPU cannot be used.")

print(f'Device: {config["device"]}')
test_gpu()



Device: cuda
CUDA is available! GPU is ready to be used.
Device name: NVIDIA GeForce RTX 2080 Ti


## Logger

In [3]:

def config_logger():
    logger = logging.getLogger()
    if logger.hasHandlers():
        logger.handlers.clear()

    now = datetime.now().strftime('%Y-%m-%d_%H:%M:%S')

    if not os.path.exists('logs'):
        os.makedirs('logs')

    logging.basicConfig(
        level=logging.INFO,
        format='[%(levelname)s: %(asctime)s] %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
        filename=f'logs/{now}.log',
    )
    
    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    formatter = logging.Formatter('[%(levelname)s: %(asctime)s] %(message)s')
    console.setFormatter(formatter)
    logger.addHandler(console)

    return logger

def log_hyperparameter(model, processor):
    config['model'] = model
    config['image_processor'] = processor
    
    hyperparameter = 'config:\n'
    for param in config:
        hyperparameter += f'{param}={config[param]}\n'

    logger.info(hyperparameter)







logger = config_logger()

# Data with torch

In [4]:
# Define your transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
     
    transforms.RandomCrop((224, 224)),
    transforms.RandAugment(num_ops=3, magnitude=7),
    transforms.ToTensor(), # Convert to tensor here!


])
# Load CSV files
train_df = pd.read_csv('dataset/train_data.csv')
test_df = pd.read_csv('dataset/test_data_public.csv')

# Load label to index mapping
with open('dataset/label_to_idx.txt', mode='r') as f:
    label2idx = f.readlines()
label2idx = dict([x.strip().split() for x in label2idx])

# Append label index to train_df
train_df['label2idx'] = train_df['labels'].apply(
    lambda labels: ','.join([label2idx[label.strip()] for label in labels.split(',')])
    #lambda labels: list(int(label2idx[label.strip()]) for label in labels.split(','))

)

 

 
class CustomDataset(Dataset):
    def __init__(self, dataframe, transform=None, train_or_test:str="train", cache_root: str = "cache"):
        self.dataframe = dataframe
        self.transform = transform
        self.train_or_test = train_or_test
        self.cache_root = cache_root

        # Create cache directory structure
        self.cache_dir = os.path.join(self.cache_root, self.train_or_test)
        os.makedirs(self.cache_dir, exist_ok=True)  # Create directory and parents if needed

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        

        img_name = self.dataframe.iloc[idx]['filename']
        # Construct cache path *mirroring* data path
        cache_path = os.path.join(self.cache_dir, img_name + ".pkl") # changed extention to .pkl
        
        if os.path.exists(cache_path):
            if self.train_or_test != 'test':

                with open(cache_path, 'rb') as f:
                    image, labels = pickle.load(f)
                return image, labels
            with open(cache_path, 'rb') as f:
                    image = pickle.load(f)
            return image
        

        img_path = f'dataset/{self.train_or_test}_data/{img_name}'
        image = Image.open(img_path).convert('RGB')
         
        if self.transform:
            image = self.transform(image)
        if self.train_or_test != 'test':
            labels = torch.zeros(config['num_of_classes'])
            for label in self.dataframe.iloc[idx]['label2idx'].split(','):
                labels[int(label)] = 1
            if self.cache_root is not None:
                with open(cache_path, 'wb') as f:
                    pickle.dump((image, labels), f)
        
            return image,  torch.tensor(labels )
        
        if self.cache_root is not None:
            with open(cache_path, 'wb') as f:
                pickle.dump((image), f)

        
        
         
         
        return image




In [5]:

# Create instances of the dataset
train_dataset = CustomDataset(train_df,  transform=transform,train_or_test="train")
#valid_dataset = CustomDataset(valid_df, transform=transform)
test_dataset = CustomDataset(test_df,  transform=transform,train_or_test="test")  # Adjust accordingly if test dataset needs transformation

# Split train_dataset into training and validation sets (e.g., 80/20 split) 
if not config['use_valid']:
    train_size = int(0.9 * len(train_dataset)) 
    valid_size = len(train_dataset) - train_size 
    train_dataset, valid_dataset = random_split(train_dataset, [train_size, valid_size])
    validloader = DataLoader(valid_dataset, batch_size=config['batch_size'], shuffle=False)

# Create DataLoaders 
 
trainloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
testloader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
 
# 



 

# Model

## resnet

In [6]:
model = AutoModelForImageClassification.from_pretrained(config['checkpoint'])
model.classifier[1] = torch.nn.Linear(in_features=model.classifier[1].in_features, out_features=config['num_of_classes'], bias=True)


if torch.cuda.device_count() > 1: 
    print("Let's use", torch.cuda.device_count(), "GPUs!") 
    model = torch.nn.DataParallel(model)

model = model.to(config['device'])

loss_func = torch.nn.BCEWithLogitsLoss() # 用於多標籤分類問題，將 sigmoid 和 binary cross entropy 合併。接受一個線性層的輸出，並將它們通過 sigmoid 函數和 binary cross entropy 函數計算損失。
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, gamma=0.1, step_size=5)

 

Let's use 2 GPUs!


# Utils

## Evaluate

In [7]:
def evaluate(model):
    model.eval()
    average_precision_scores = []
    losses = []

    all_sample_preds = []
    all_sample_labels = []

    with torch.no_grad():
        for imgs, labels in validloader:
            imgs, labels = imgs.to(config['device']), labels.to(config['device'])
            outputs = model(imgs)['logits']

            loss = loss_func(outputs, labels)
            losses.append(loss.detach().item())

            preds = torch.sigmoid(outputs)

            all_sample_preds.append(preds)
            all_sample_labels.append(labels)

    all_sample_preds = torch.cat(all_sample_preds)
    all_sample_labels = torch.cat(all_sample_labels)

    for c in range(config['num_of_classes']):
        ap = average_precision_score(all_sample_labels[:, c].cpu().numpy(), all_sample_preds[:, c].cpu().numpy())
        average_precision_scores.append(ap)

    average_precision_scores.append(ap)

    return np.mean(average_precision_scores), np.mean(losses)

## Save model
## Load model

In [8]:
def load_model(model, path):
    model.load_state_dict(torch.load(path))
    #model.load_state_dict(torch.load(path, map_location=config['device']), weights_only=True)
    return model
def save_model(model):
    if not os.path.exists('models'):
        os.makedirs('models')

    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    logger.info(f'Saving model to models/{now}.pt')

    torch.save(model.state_dict(), f'models/{now}.pt')

# Training

In [ ]:
def training(model):
    epochs = config['epochs']
    # training_losses = []
    patient = 0
    best_mAP =0

    for epoch in tqdm(range(epochs)):
        model.train()
        epoch_loss = []

        for imgs, labels in trainloader:
             
            imgs= imgs.to(config['device'])
            
            
              
            labels=  labels.to(config['device'] ) 
            
            
            output = model(imgs)['logits']
            loss = loss_func(output, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss.append(loss.detach().item())
        
        scheduler.step()
        

        # Early stopping with mAP
        if not config['use_valid']:
            mAP, _ = evaluate(model)
            logger.info(f'Epoch {epoch+1}/{epochs}, loss: {np.mean(epoch_loss)}, eval mAP: {mAP}')

            if epoch>7 and  mAP > best_mAP:
                best_mAP = mAP
                patient = 0  # Reset patient counter on improvement
            else:
                patient += 1

            if patient == config['patient']:
                logger.info(f'Early stopping in epoch {epoch+1} due to no improvement in mAP for {patient} epochs')
                return
        
        #Early stopping end with mAP
        logger.info(f'Epoch {epoch+1}/{epochs}, loss: {np.mean(epoch_loss)}')
        #training_losses.append(np.mean(epoch_loss))
        # # Early stopping start
        # if epoch > 5 and training_losses[-1] > training_losses[-2]:
        #     patient += 1
        # else:
        #     patient = 0
        
        # if patient == config['patient']:
        #     logger.info(f'Early stopping in epoch {epoch+1}')
        #     return
        # Early stopping end
        
        
        # Evalation start
        # mAP, eval_loss = evaluate(model)
        # logger.info(f'Epoch {epoch+1}/{epochs}, eval mAP: {mAP}, eval loss: {eval_loss}')
        # Evaluation end
        

# Start training

In [ ]:
try:
    log_hyperparameter(config['checkpoint'], config['checkpoint'])
    model = load_model(model, f'models/2024-12-16 17:33:25.pt')
    training(model)
    save_model(model)
except Exception as e:
    logger.error(f'An error occured: {e}\n Traceback: \n {str(traceback.format_exc())}')

# Start Inference

## Inference

In [12]:
def inference(model):
    model.eval()
    predictions = []

    with torch.no_grad():
        for imgs in tqdm(testloader):
            imgs = imgs.to(config['device'])
            outputs = model(imgs)['logits']
            preds = torch.sigmoid(outputs)
            predictions.append(preds.cpu().numpy())

    return np.concatenate(predictions)

In [13]:
model_name = '2024-12-20 23:37:52.pt'
resnet = load_model(model, f'models/{model_name}')
predictions = inference(resnet)

/tmp/ipykernel_2058405/3853542169.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path))
100%|██████████████████████████████████████████

# Summit to kaggle

In [14]:
result = pd.DataFrame(predictions, columns=[f'class_{i}_prob' for i in range(79)])
result.insert(0, 'filename', test_df['filename'])

now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
result.to_csv(f'./submissions/{now}_{config['model'].split("/")[1]}_{config['epochs']}.csv', index=False)

print('Done!')

Done!
